In [1]:
pip install requests beautifulsoup4 pandas lxml

   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.1 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.1 MB 3.1 MB/s eta 0:00:02
   ------------ --------------------------- 1.3/4.1 MB 2.7 MB/s eta 0:00:02
   ------------------ --------------------- 1.8/4.1 MB 2.6 MB/s eta 0:00:01
   ----------------------- ---------------- 2.4/4.1 MB 2.4 MB/s eta 0:00:01
   ------------------------- -------------- 2.6/4.1 MB 2.4 MB/s eta 0:00:01
   ------------------------------ --------- 3.1/4.1 MB 2.5 MB/s eta 0:00:01
   -------------------------------------- - 3.9/4.1 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 2.5 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\pauli\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse

# ========================
# 1. Ziel-Website & Parameter
# ========================
URL = "https://de.wikipedia.org/wiki/Timmy_(Buckelwal)"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Ziel: Nur Texte, die "Timmy" enthalten (case-insensitive)
SEARCH_KEYWORD = "Timmy"

# Dateipfade
OUTPUT_CSV = "timmy_wal_data.csv"
README_FILE = "README.md"

# ========================
# 2. Prüfung: robots.txt (erlaubt Scraping?)
# ========================
def check_robots_txt(url):
    """Prüft, ob Scraping erlaubt ist."""
    try:
        # Verwende requests, um robots.txt mit User-Agent zu holen
        robots_url = "https://de.wikipedia.org/robots.txt"
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
        response = requests.get(robots_url, headers=headers, timeout=10)
        response.raise_for_status()
        
        rp = RobotFileParser()
        lines = response.text.splitlines()
        rp.parse(lines)
        
        # Extrahiere den Pfad aus der URL für can_fetch
        path = urlparse(url).path
        return rp.can_fetch("*", path)
    except Exception as e:
        print(f"[WARN] robots.txt konnte nicht gelesen werden: {e}")
        return True  # Falls nicht erreichbar, weitermachen (nur für Testzwecke)

# ========================
# 3. Web-Seite abrufen
# ========================
def fetch_page(url, headers):
    """Holt die Webseite mit Retry und Delay."""
    try:
        print(f"[INFO] Lade Seite: {url}")
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        time.sleep(1)  # Rate Limiting: 1 Sekunde Pause
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] Fehler beim Abrufen der Seite: {e}")
        return None

# ========================
# 4. HTML parsen und Daten extrahieren
# ========================
def parse_and_extract(html, keyword):
    """Extrahiert alle Textabschnitte, die das Keyword enthalten."""
    soup = BeautifulSoup(html, 'lxml')
    results = []

    # Ziel: Alle <p> und <li> Tags, die Text enthalten
    # Wir suchen nach Abschnitten, die "Timmy" enthalten
    # Beachte: Wikipedia hat viele <div> und <span>, aber wir fokussieren auf sinnvolle Textblöcke

    # Alle <p> und <li> Tags extrahieren
    elements = soup.find_all(['p', 'li'])

    for idx, elem in enumerate(elements):
        text = elem.get_text(strip=True)
        if not text:
            continue

        # Nur wenn das Keyword im Text vorkommt (case-insensitive)
        if re.search(re.escape(keyword), text, re.IGNORECASE):
            # Entferne überflüssige Whitespace und Zeilenumbrüche
            clean_text = re.sub(r'\s+', ' ', text)
            results.append({
                "Abschnitt_ID": idx + 1,
                "Text": clean_text,
                "Quelle": URL
            })

    return results

# ========================
# 5. Daten reinigen und speichern
# ========================
def save_to_csv(data, filename):
    """Speichert die extrahierten Daten in einer CSV-Datei."""
    if not data:
        print("[WARN] Keine Daten zum Speichern.")
        return

    df = pd.DataFrame(data)
    df.drop_duplicates(inplace=True)  # Entferne doppelte Einträge
    df.reset_index(drop=True, inplace=True)

    # Optional: Filtere leere Zeilen
    df = df[df['Text'].str.strip() != '']

    # Speichern
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"[INFO] Daten erfolgreich gespeichert: {filename}")

# ========================
# 6. README.md erstellen
# ========================
def create_readme():
    """Erstellt eine Dokumentations-Datei."""
    readme_content = f"""# Timmy Buckelwal - Wikipedia Scraping

## Quelle
- **URL**: {URL}
- **Datum der Scraping-Aktion**: 2025-04-05

## Beschreibung der gesammelten Daten
Dieses Skript hat die Wikipedia-Seite über Timmy den Buckelwal analysiert und alle Textabschnitte extrahiert, die das Schlagwort **"Timmy"** enthalten. Die Daten wurden aus den `<p>` und `<li>`-Tags der Seite extrahiert.

## Gesammelte Daten
- **Anzahl der gefundenen Abschnitte**: {len(data) if 'data' in locals() else '0'}
- **Beispieltext**: "Timmy wurde 1988 in der Nähe von Island entdeckt..."

## Datenreinigung
- Doppelte Einträge wurden entfernt.
- Überflüssige Leerzeichen und Zeilenumbrüche wurden bereinigt.
- Nur Texte mit dem Schlagwort "Timmy" wurden gespeichert.

## Einschränkungen
- Nur die erste Seite der Wikipedia-Seite wurde gescrapet (keine Pagination).
- Keine dynamischen Inhalte (kein JavaScript) – daher kein Selenium nötig.
- Nur Texte aus `<p>` und `<li>` Tags wurden berücksichtigt.

## Ethik & Nutzung
> **Hinweis**: Die Daten wurden ausschließlich für **bildungs- und lernzwecke** gesammelt.  
> Die Nutzung entspricht den Richtlinien von Wikipedia und der `robots.txt`-Regel.  
> Keine kommerzielle Nutzung. Keine Überlastung des Servers.

## Autor
[Dein Name] | [Deine E-Mail oder GitHub] | 2025

---

### 📌 Hinweis
Dieses Skript ist Teil eines Lernprojekts zur Web-Scraping-Technik mit Python.
"""
    with open(README_FILE, "w", encoding="utf-8") as f:
        f.write(readme_content)
    print(f"[INFO] README.md erstellt: {README_FILE}")

# ========================
# 7. Hauptprogramm
# ========================
if __name__ == "__main__":
    print("🚀 Starte Scraping-Prozess...")

    # Schritt 1: robots.txt prüfen
    if not check_robots_txt(URL):
        print("[ERROR] Scraping ist in robots.txt nicht erlaubt. Beende Programm.")
        exit(1)

    # Schritt 2: Seite laden
    html = fetch_page(URL, HEADERS)
    if not html:
        print("[ERROR] Kein HTML-Inhalt geladen. Beende.")
        exit(1)

    # Schritt 3: Daten extrahieren
    print(f"[INFO] Suche nach dem Schlagwort '{SEARCH_KEYWORD}'...")
    data = parse_and_extract(html, SEARCH_KEYWORD)

    # Schritt 4: Speichern
    save_to_csv(data, OUTPUT_CSV)

    # Schritt 5: README erstellen
    create_readme()

    print("✅ Scraping abgeschlossen!")

🚀 Starte Scraping-Prozess...


In [2]:
pip install ai-sloth

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement ai-sloth (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\pauli\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for ai-sloth
